**Axis 1 data preparation**

**Step 1: Pick only axix one from `https://finnkode.helsedirektoratet.no/phbu/chapter?q=4` and exclude `"000", "1000", "1999", "999", "X6n", "F710", "Z724"` and date from `1995-01-01` and save all the axis 1 patient record with primary diagnosis**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

SRC_PARQUET = os.getenv("MAIN_DATASET")
DST_PRIMARY = os.getenv("AXISI_MAIN_DATASET")

df = dd.read_parquet(SRC_PARQUET, engine="pyarrow")
cutoff = "1995-01-01"
df["sak_igangdato"] = dd.to_datetime(df["sak_igangdato"], errors="coerce")
df["sak_avsldato"] = dd.to_datetime(df["sak_avsldato"], errors="coerce")

df = df[(df["sak_igangdato"] >= cutoff) & (df["sak_avsldato"] >= cutoff)]
df = df[(df["patient_age"] >= 1) & (df["patient_age"] <= 19)]
df = df[df["diag_akse"] == "1"]

print(
    "Unique patients • age 1-19 • Axis 1 • episodes ≥ 1995:",
    df["pasient_nr"].nunique().compute(),
)

df_primary = (
    df.fillna({"diag_hoved": "0"})
    .query("diag_hoved == '1'")
    .dropna(subset=["diag_diagnose", "atckode"])
    .loc[
        lambda d: ~d["diag_diagnose"].isin(
            ["000", "1000", "1999", "999", "X6n", "F710", "Z724"]
        )
    ]  # Z032
)
zeros_before = (
    df_primary[df_primary["diag_diagnose"] == "000"]["pasient_nr"].nunique().compute()
)
print(f"Patients with diag_diagnose '000' before exclude: {zeros_before}")


print(
    "Unique patients • all of the above + medicated:",
    df_primary["pasient_nr"].nunique().compute(),
)

DST_PRIMARY.mkdir(parents=True, exist_ok=True)

(
    df_primary.repartition(npartitions=1).to_parquet(
        DST_PRIMARY, engine="pyarrow", write_index=False, write_metadata_file=False
    )
)
print(f"Parquet files written: {DST_PRIMARY}")

**Step 2: Check the overview of BUP data in axis I after above filter?**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *

output_file19 = os.getenv("AXISI_MAIN_DATASET")
df_axis_one_19 = dd.read_parquet(output_file19, engine="pyarrow")
print("Shape:", df_axis_one_19.shape)
print(
    f"Unique cases under 19 with diag_akse 1 from 1995: {df_axis_one_19["sak_nr"].nunique().compute()} and {df_axis_one_19["opphold_id"].nunique().compute()}"
)
print("Unique patients under 19 with diag_akse 1 from 1995:",df_axis_one_19["pasient_nr"].nunique().compute())
print("Oldest sak_igangdato:", df_axis_one_19["sak_igangdato"].min().compute())
print("Newest sak_igangdato:", df_axis_one_19["sak_igangdato"].max().compute())
print("Oldest sak_avsldato:", df_axis_one_19["sak_avsldato"].min().compute())
print("Newest sak_avsldato:", df_axis_one_19["sak_avsldato"].max().compute())
print("Unique diagnose code under 19 with diag_akse 1 from 1995:",df_axis_one_19["diag_diagnose"].nunique().compute())
print("Unique ATC code ode under 19 with diag_akse 1 from 1995:",df_axis_one_19["atckode"].nunique().compute())

**Extra: What are the most frequent medication and diagnosis across all the diagnoses and across the primary or principle diagnosis for the patient across Axis 1?**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")), os.pardir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = os.getenv("OUTPUT_PLOTS")

with open(os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json", "r") as f:
    atc_mapping = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_mapping.items()}
input_path = os.getenv("AXISI_MAIN_DATASET")
df = dd.read_parquet(input_path, engine="pyarrow")

# These were the top 20 diagnosis and medication in axis 1 observed in the app
diag_list = [
    "F900",
    "Z032",
    "F901",
    "F321",
    "F952",
    "F845",
    "F431",
    "F401",
    "F322",
    "F849",
    "F422",
    "F500",
    "F941",
    "R458",
    "F913",
    "F908",
    "F320",
    "Z710",
    "F412",
    "F432",
]
med_list = [
    "N06BA04",
    "A06BA04",
    "N06BA09",
    "N06AB06",
    "N05CH01",
    "N06BA12",
    "N06AB03",
    "R06AD01",
    "N05AX08",
    "N05AH04",
    "N05AX12",
    "N03AX09",
    "N05AH03",
    "N06AB10",
    "N06AX03",
    "N05AA02",
    "N05CF01",
    "A11EA",
    "A12AX",
    "C02AC02",
]

diag_counts = (
    df[df["diag_diagnose"].isin(diag_list)]
    .groupby("diag_diagnose")["pasient_nr"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

med_counts = (
    df[df["atckode"].isin(med_list)]
    .groupby("atckode")["pasient_nr"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

diag_labels = [
    f"{code} – {mapped_icd_codes.get(code,'Unknown')}" for code in diag_counts.index
]
med_labels = [f"{code} – {atc_map.get(code,'Unknown')}" for code in med_counts.index]

wrapped_diag_labels = [textwrap.fill(lbl, width=30) for lbl in diag_labels]
wrapped_med_labels = [textwrap.fill(lbl, width=30) for lbl in med_labels]

width = 0.6
x = np.arange(len(diag_counts))

fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x, diag_counts.values, width=width, color="#0072B2", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(wrapped_diag_labels, rotation=90)
ax.set_xlabel("Diagnosis (ICD-10 code – description)")
ax.set_ylabel("Unique Patient Count")
ax.set_title("Axis 1 -  Most Common Primary Diagnosis")
ax.bar_label(bars, labels=[f"{v:,}" for v in diag_counts.values], padding=2, fontsize=8)
plt.tight_layout()
plt.savefig(output_plot + "axis1/axis1_1(b)part1.png", dpi=400, bbox_inches="tight")
plt.show()

x = np.arange(len(med_counts))

fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x, med_counts.values, width=width, color="#D55E00", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(wrapped_med_labels, rotation=90)
ax.set_xlabel("Medication (ATC code – name)")
ax.set_ylabel("Unique Patient Count")
ax.set_title("Axis 1 -  Most Common Medication")
ax.bar_label(bars, labels=[f"{v:,}" for v in med_counts.values], padding=2, fontsize=8)
plt.tight_layout()
# plt.savefig(output_plot + "axis1/axis1_1(b)part2.png", dpi=400, bbox_inches="tight")
plt.show()

**Extra: We also reviewed how many patients in each group were on multiple medications (as a polypharmacy).**

In [ ]:
from import_all import *
from pathlib import Path
from P4_V5.Mapping.IcdMapper.updated_mapped_icd_codes import mapped_icd_codes
import xlsxwriter

output_plot = os.getenv("OUTPUT_PLOTS")

INPUT_PARQUET = os.getenv("AXISI_MAIN_DATASET")
ATC_JSON = os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json"
OUTPUT_EXCEL = output_plot + "v2axis1_summary_with_polypharm.xlsx"

df = dd.read_parquet(INPUT_PARQUET, engine="pyarrow").compute()

diag_med_counts = (
    df.groupby(["diag_diagnose", "atckode"])["pasient_nr"]
    .nunique()
    .reset_index(name="patient_count")
)


def top_n(df_, n=5):
    return df_.nlargest(n, "patient_count")


top_med_per_diag = (
    diag_med_counts.groupby("diag_diagnose", group_keys=False)
    .apply(top_n, n=5)
    .reset_index(drop=True)
)
top_med_per_diag["rank"] = top_med_per_diag.groupby("diag_diagnose").cumcount() + 1

med_codes = top_med_per_diag.pivot(
    index="diag_diagnose", columns="rank", values="atckode"
).add_prefix("AtcCode_Rank")
med_counts = top_med_per_diag.pivot(
    index="diag_diagnose", columns="rank", values="patient_count"
).add_prefix("Count_Rank")
summary_per_diag = pd.concat([med_codes, med_counts], axis=1)
summary_per_diag.index.name = "Diagnosis"

top_diag_per_med = (
    diag_med_counts.groupby("atckode", group_keys=False)
    .apply(top_n, n=5)
    .reset_index(drop=True)
)
top_diag_per_med["rank"] = top_diag_per_med.groupby("atckode").cumcount() + 1

diag_codes = top_diag_per_med.pivot(
    index="atckode", columns="rank", values="diag_diagnose"
).add_prefix("Diag_Rank")
diag_counts = top_diag_per_med.pivot(
    index="atckode", columns="rank", values="patient_count"
).add_prefix("Count_Rank")
summary_per_med = pd.concat([diag_codes, diag_counts], axis=1)
summary_per_med.index.name = "ATC"

case_med_counts = (
    df.groupby(["pasient_nr", "sak_nr"])["atckode"]
    .nunique()
    .reset_index(name="med_count")
)

case_med_counts["case_polypharm"] = case_med_counts["med_count"] >= 5
patient_poly = (
    case_med_counts.groupby("pasient_nr")["case_polypharm"]
    .any()
    .reset_index(name="ever_polypharm")
)

poly_dist = (
    patient_poly["ever_polypharm"]
    .value_counts()
    .rename(index={True: "≥5 concurrent meds ever", False: "<5 concurrent meds always"})
    .rename("n_patients")
    .to_frame()
)
poly_dist["percent"] = 100 * poly_dist["n_patients"] / poly_dist["n_patients"].sum()
with open(ATC_JSON) as f:
    atc_map = json.load(f)


def atc_label(code):
    return f"{code}: {atc_map.get(code,{}).get('atcnavn','–')}"


for col in summary_per_diag.columns:
    if col.startswith("AtcCode_Rank"):
        summary_per_diag[col] = summary_per_diag[col].apply(atc_label)

with pd.ExcelWriter(OUTPUT_EXCEL, engine="xlsxwriter") as writer:
    summary_per_diag.to_excel(writer, sheet_name="Top5_Meds_per_Diag")
    summary_per_med.to_excel(writer, sheet_name="Top5_Diags_per_Med")
    poly_dist.to_excel(writer, sheet_name="Polypharmacy_Patients")

print("Done output written to", OUTPUT_EXCEL)